In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime
import openpyxl
import plotly.express as px
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import os
import urllib.parse

# Almacenamos las credenciales de acceso que nos provee Docta una vez creamos una cuenta ("client_id" y "client_secret") en un archivo .env
# Cargamos las variables del archivo .env
load_dotenv()  

CLIENT_SECRET = os.getenv("CLIENT_SECRET")
CLIENT_ID = os.getenv("CLIENT_ID")




In [ ]:
url = "https://api.doctacapital.com.ar/api/v1/auth/token"

payload = {
    "grant_type": "client_credentials",
    "client_id": f"{urllib.parse.quote(CLIENT_ID)}",
    "client_secret": f"{urllib.parse.quote(CLIENT_SECRET)}"
}

headers = {"Content-Type": "application/json"}

response = requests.post(url, json=payload, headers=headers)

# Convertimos la respuesta a JSON
response_json = response.json()

bearer_token = response_json["access_token"]


In [ ]:
# Obtenemos información de un numero instrumentos de un mismo tipo.
instrument = "DOLLAR_LINKED"

url = f"https://api.doctacapital.com.ar/api/v1/bonds/instruments?limit=50&sub_asset_class={instrument}"

payload = ""
headers = {
  'Authorization': f'Bearer {bearer_token}'
}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)


{"data":[{"ticker":"D30A6","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"TZV6C","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"TZV6D","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"TZV26","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"D16E6","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"D27F6","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"},{"ticker":"D30E6","asset_class":"BOND","sub_asset_class":"DOLLAR_LINKED"}],"pagination":{"limit":50,"has_next":false,"next_cursor":null,"total_items":7},"filters":{"sub_asset_class":"DOLLAR_LINKED"},"metadata":{"total_records":7}}


In [ ]:
# Obtenemos datos generales de un bono
bono = "AN29"

url_bono = f"https://api.doctacapital.com.ar/api/v1/bonds/instruments/{bono}"


payload = ""
headers = {
  'Authorization': f'Bearer {bearer_token}'
}

response = requests.request("GET", url_bono, headers=headers, data=payload)

print(response.text)

{"data":[{"ticker":"AN29","name":"Soberano HD 2029 U$S 6.5% (AN29)","asset_class":"BOND","sub_asset_class":"HARD_DOLLAR","sector":null,"issuer":null,"law":null}],"metadata":{"total_records":1}}


In [ ]:
# Datos de rendimiento intra diario de un bono.
url_yield = f"https://api.doctacapital.com.ar/api/v1/bonds/yields/{bono}/intraday"

payload = ""
headers = {
  'Authorization': f'Bearer {bearer_token}'
}

response_yield = requests.request("GET", url_yield, headers=headers, data=payload)

print(response_yield.text)

{"ticker":"AN29","data":[{"tir":0.09418569140654638,"tna":0.10764231485320458,"tna_30_360":0.09034884962608292,"tem":0.007529070802173576,"duration":3.434457496108851,"dtm":1410,"margen":null}],"metadata":{"total_records":1,"sub_asset_class":"HARD_DOLLAR"}}


In [ ]:
# Obtener el calendario de pagos de varios bonos.
# cash_flow almacena una lista de diccionarios (una respuesta por ticker).
bonos = ["AN29", "AL41", "AE38"]
cash_flow = []

for ticker in bonos:
    url_flujo = f"https://api.doctacapital.com.ar/api/v1/bonds/analytics/{ticker}/cashflow"
    response_flujo = requests.get(
        url_flujo,
        headers={"Authorization": f"Bearer {bearer_token}"},
        timeout=30,
    )
    response_flujo.raise_for_status()
    cash_flow.append(response_flujo.json())

print(f"Flujos obtenidos: {len(cash_flow)} bonos")


## Visualización del calendario de pagos

La última celda acepta `cash_flow` como lista de diccionarios o de cadenas JSON (como el ejemplo compartido). Dibuja un gráfico independiente por bono: barras apiladas de `adj_capital` y `adj_interest_amount`, y puntos con el `cash_flow` total informado por la API. Se usan los campos ajustados porque, en bonos amortizables, `interest_amount` puede diferir del interés efectivo incluido en el flujo.

Se conservan todos los pagos, incluidos los históricos. Los montos corresponden a las `nominal_units` de cada respuesta, no al valor de una cartera. No se suman bonos ni se presupone una moneda común. Para ejecutar solo la visualización, carga previamente tu lista en `cash_flow`; no requiere nuevas llamadas a la API.


In [ ]:
import plotly.graph_objects as go

if not isinstance(cash_flow, list) or not cash_flow:
    raise ValueError("cash_flow debe ser una lista no vacía de respuestas de Docta.")

figuras_flujos = []
tablas_flujos = []

for indice, entrada in enumerate(cash_flow, start=1):
    flujo = json.loads(entrada) if isinstance(entrada, str) else entrada
    if not isinstance(flujo, dict) or not flujo.get("ticker"):
        raise ValueError(f"La respuesta {indice} debe ser un diccionario con ticker y data.")
    ticker = flujo["ticker"]
    pagos = flujo.get("data")
    if not isinstance(pagos, list):
        raise ValueError(f"{ticker}: data debe ser una lista de pagos.")
    if not pagos:
        print(f"{ticker}: sin pagos para graficar.")
        continue

    df = pd.DataFrame(pagos)
    campos = ["payment_date", "adj_capital", "adj_interest_amount", "cash_flow"]
    faltantes = set(campos) - set(df.columns)
    if faltantes:
        raise ValueError(f"{ticker}: faltan los campos {sorted(faltantes)}.")
    df["payment_date"] = pd.to_datetime(df["payment_date"], errors="raise")
    for campo in campos[1:]:
        df[campo] = pd.to_numeric(df[campo], errors="raise")
    if df[campos].isna().any().any():
        raise ValueError(f"{ticker}: hay fechas o importes nulos.")
    df = df.sort_values("payment_date").reset_index(drop=True)
    df["ticker"] = ticker
    tablas_flujos.append(df)

    nominales = flujo.get("metadata", {}).get("nominal_units")
    base = f"por {nominales} unidades nominales" if nominales is not None else "base nominal no informada"
    fig = go.Figure()
    for campo, nombre, color in [
        ("adj_capital", "Amortización ajustada", "#2563eb"),
        ("adj_interest_amount", "Intereses ajustados", "#14b8a6"),
    ]:
        fig.add_bar(
            x=df["payment_date"], y=df[campo], name=nombre,
            marker_color=color,
            hovertemplate="%{x|%d/%m/%Y}<br>%{y:,.4f}<extra>%{fullData.name}</extra>",
        )
    fig.add_scatter(
        x=df["payment_date"], y=df["cash_flow"], mode="markers",
        name="Flujo total informado", marker=dict(color="#0f172a", size=7),
        hovertemplate="%{x|%d/%m/%Y}<br>Total: %{y:,.4f}<extra></extra>",
    )
    fig.update_layout(
        title=f"{ticker} · Calendario de pagos ({base})",
        barmode="relative", template="plotly_white", height=450,
        xaxis_title="Fecha de pago", yaxis_title="Importe en la unidad monetaria del bono",
        legend_title_text="Componente", hovermode="x unified",
    )
    fig.update_xaxes(type="date", tickformat="%d/%m/%Y", rangeslider_visible=True)
    figuras_flujos.append(fig)

# Tabla reutilizable para análisis o exportación, sin modificar cash_flow.
df_flujos = pd.concat(tablas_flujos, ignore_index=True) if tablas_flujos else pd.DataFrame()
for fig in figuras_flujos:
    fig.show()
